# S6E11 Ensemble: hill climb vs logistic stack on the saved OOF library

This notebook blends the accepted library (`experiments/library.txt`) from the saved out-of-fold predictions, with no retraining. Both blenders are scored by **nested** CV on the same `folds.npy` folds as the members: fold k is scored by a blend fitted on the other four. The better one writes a rank-scaled submission. Its score must equal the last `STACK` row that `experiments/stack.py` logged for the same library.

In [1]:
import json
import re
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from kvbiii_ml.modeling.optimization.oof_ensembling import hill_climb, logistic_stack
from scipy.stats import rankdata

Paths and names shared by every cell. `TOLERANCE` is the rounding of the logged scores (5 decimals).

In [2]:
OOF_DIR = Path("oof") / "full"
LIBRARY_FILE = Path("experiments") / "library.txt"
LOG_FILE = Path("experiments.md")
DATA_DIR = Path("data")
SUBMISSION_DIR = Path("submissions")
TARGET = "is_fraud"
REFIT_SUFFIX = "_refit"
TOLERANCE = 5e-6

## 1. Library

The same loaders as the Advanced notebook. Members must share rows and folds, which `member_matrix` checks before stacking them into the `(n, m)` OOF and test matrices.

In [3]:
def load_member(name: str) -> dict:
    """Read one full-mode member `.npz` into a dict with its JSON config decoded."""
    with np.load(OOF_DIR / f"{name}.npz") as data:
        member = {key: data[key] for key in data.files}
    member["config"] = json.loads(str(member["config"]))
    return member


def run_log() -> pd.DataFrame:
    """Parse the auto-appended run log table at the end of `experiments.md` into a DataFrame."""
    text = LOG_FILE.read_text().split("## Run log", 1)[1]
    rows = [[part.strip() for part in line.strip().strip("|").split("|")] for line in text.splitlines() if re.match(r"\| \d\d-\d\d ", line)]
    return pd.DataFrame([row[:7] + [""] * (7 - len(row[:7])) for row in rows], columns=["time", "entry", "mode", "result", "detail", "extra", "verdict"])


def member_matrix(names: list[str]) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Stack members into (oof, test, folds, rows) after checking they share rows and folds."""
    members = [load_member(name) for name in names]
    for name, member in zip(names, members, strict=True):
        if not (np.array_equal(member["rows"], members[0]["rows"]) and np.array_equal(member["fold"], members[0]["fold"])):
            raise ValueError(f"{name} does not share rows/folds with {names[0]}.")
    return np.column_stack([m["oof"] for m in members]), np.column_stack([m["test"] for m in members]), members[0]["fold"], members[0]["rows"]

Load the library, the target in train row order and the matrices. The single-model CVs show the starting point the blend has to beat.

In [4]:
library = LIBRARY_FILE.read_text().split()
oof, test, folds, rows = member_matrix(library)
y = pd.read_csv(DATA_DIR / "train.csv", usecols=[TARGET])[TARGET].astype(int).to_numpy()[rows]
singles = pd.Series({name: load_member(name)["fold_auc"].mean() for name in library}, name="single CV").sort_values(ascending=False)
singles.to_frame().style.format("{:.5f}")

,single CV
glm_boost,0.95026
xgb_d2_te,0.95021
baseline,0.94666


## 2. Blenders

**Hill climb**: Caruana selection with replacement on ranks, non-negative weights summing to 1, taken at the best iteration. **Logistic stack**: logistic regression on logit-ranks, with C chosen per outer fold by inner 5-fold CV over 3 seeds. Each takes a few minutes on 700k rows.

In [5]:
results = {"hill_climb": hill_climb(oof, y, folds, test, space="rank"), "logistic_stack": logistic_stack(oof, y, folds, test)}
comparison = pd.DataFrame({method: {"nested CV": r.score, **{f"fold {k}": s for k, s in enumerate(r.fold_scores)}, "vs best single": r.score - singles.max()} for method, r in results.items()}).T
comparison.style.format("{:+.5f}", subset=["vs best single"]).format("{:.5f}", subset=[c for c in comparison.columns if c != "vs best single"])

,nested CV,fold 0,fold 1,fold 2,fold 3,fold 4,vs best single
hill_climb,0.95065,0.94870,0.95096,0.94702,0.95067,0.95589,+0.00039
logistic_stack,0.95067,0.94874,0.95098,0.94703,0.95068,0.95589,+0.00041


### 2.1 Weights

Hill-climb weights sum to 1; stack weights are logistic coefficients on logit-ranks (signed, not normalised). A member near zero in both is a candidate for the pruning pass.

In [6]:
pd.DataFrame({method: r.weights for method, r in results.items()}, index=library).style.format("{:.3f}")

,hill_climb,logistic_stack
xgb_d2_te,0.341,0.504
glm_boost,0.512,0.680
baseline,0.146,0.176


### 2.2 Check against `stack.py`

The last plain `STACK` row per method in the run log comes from `experiments/stack.py` on the current library. The notebook must reproduce it.

In [7]:
log_lines = [line for line in LOG_FILE.read_text().splitlines() if f"[{len(library)} members]" in line and "| STACK " in line and REFIT_SUFFIX not in line]
logged = {method: float([line for line in log_lines if f"STACK {method} " in line][-1].split("|")[4]) for method in results}
for method, r in results.items():
    assert abs(r.score - logged[method]) < TOLERANCE, (method, r.score, logged[method])
logged

{'hill_climb': 0.95065, 'logistic_stack': 0.95067}

## 3. Submission

The better blender by nested CV writes its test blend, rank-scaled to (0, 1], in the sample-submission order.

In [8]:
method, best = max(results.items(), key=lambda item: item[1].score)
submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
submission[TARGET] = rankdata(best.test) / len(best.test)
path = SUBMISSION_DIR / f"{datetime.now():%m%d_%H%M}_notebook_{method}_{len(library)}m.csv"
submission.to_csv(path, index=False)
print(f"{method}: nested CV {best.score:.5f} -> {path} ({len(submission)} rows)")

logistic_stack: nested CV 0.95067 -> submissions/1127_1402_notebook_logistic_stack_3m.csv (299984 rows)
